## Imports

The imports here define the core building blocks used in a CmmDFT adsorption calculation:
- `NanoporousHost` and `SphericalLJGuest` / `DualModelGuest` specify the host framework and guest species
- `Program` manages the grand-canonical DFT setup
- `Calculator` extracts observables such as adsorption loadings
- `PCSAFTEOS` and `PengRobinsonMixEOS` provide the equation-of-state models used in the free-energy functional
- the parameter dictionaries provide reusable force-field data for guests and host atoms


In [ ]:
import numpy as np

from cmmdft.units_constants import *

from cmmdft.system import NanoporousHost, SphericalLJGuest, DualModelGuest, EmptyHost, GuestMixture
from cmmdft.program import Program
from cmmdft.solver import Picard, Anderson, QuasiNewton
from cmmdft.calculator import Calculator
from cmmdft.plotter import Plotter

This notebook demonstrates a complete adsorption workflow in CmmDFT for a single guest and then for a guest mixture. The overall steps are:

1. define the host and guest model(s)
2. choose the EOS and free-energy terms
3. set the grid and temperature
4. solve the adsorption problem for a set of pressures
5. extract the resulting loadings or isotherms

The example is intentionally structured as a practical blueprint: it shows how the host/guest definitions, the program setup, and the solver configuration fit together in a realistic calculation.


In [ ]:
LJ_pars_dict = {'CH4':  (16.031*amu, 3.75*angstrom, 0.29411*kcalmol),
                'C2H6': (30.047*amu, 4.41*angstrom, 0.46209*kcalmol),
                'C2H4': (28.031*amu, 4.24*angstrom, 0.42730*kcalmol)
               }

SAFT_pars_dict = {'CH4' : (16.031*amu, 3.70051*angstrom, 150*boltzmann, 1.0),
                  'C2H6': (30.047*amu, 3.51681*angstrom, 191.45389*boltzmann, 1.60686),
                  'C2H4': (28.031*amu, 3.40769*angstrom, 177.32571*boltzmann, 1.59196),
                  'C3H6': (42.047*amu, 3.53847*angstrom, 207.73187*boltzmann, 1.95306),
                  'C3H8': (44.063*amu, 3.6244*angstrom, 209.08586*boltzmann, 1.98602),
                  'CO2' : (43.99*amu,  2.57855*angstrom, 153.31864*boltzmann, 2.53096)}

## System setup

This section defines the framework and the adsorbate. The host is the porous material (here, ZIF-8), and the guest is the molecule whose adsorption is being studied. The file paths point to the host and guest data files used to build the interaction models.

The guest model is a `DualModelGuest`, which combines a hard-sphere repulsive description with a molecular interaction model. This is common when using PC-SAFT-based free-energy functionals. The `temperature` and `pressures` arrays define the conditions for the isotherm calculation.


In [ ]:
guestname = 'C2H6_UA'
mass, sigma, epsilon = LJ_pars_dict['C2H6']

guestchk = 'InputFiles/Guest/C2H6/struct_CA.chk'
guestpar = 'InputFiles/Guest/C2H6/pars_vdw_CA_trappe.txt'
guest = DualModelGuest(guestname, mass, sigma, epsilon, struct=guestchk, par=guestpar, ffname='TRAPPE-DUAL', hs_def='exp')

hostname = 'ZIF8'
hostchk = 'InputFiles/Host/ZIF8/ZIF8_struct.chk'
hostpar = 'InputFiles/Host/ZIF8/pars_vdw_AMBER.txt'
host = NanoporousHost(hostname, hostchk, hostpar)

ff_suffix = 'AMBER_TRAPPE'
funct_suffix = 'tWBII_PCSAFT'

grid_suffix = '0_50'

In [ ]:
# thermodynamic properties
# We run the adsorption isotherm at one temperature and over a pressure range.
# In a grand-canonical DFT workflow, the program solves the density profile at each pressure
# and converts the result into an equilibrium loading.
temperature = 300*kelvin
pressures = np.linspace(0.1,50,10)*bar


In [ ]:
# program initialization
# The Program object is the central configuration container for the adsorption calculation.
# It links the host, guest, EOS, grid, free-energy terms, and solver into a single workflow.
program = Program(prefix=f'OutputFiles/AdsorptionExample', hostname=hostname, guestname=guestname, ff_suffix=ff_suffix, funct_suffix=funct_suffix, grid_suffix=grid_suffix)

program.set_system(host, guest)
program.set_grid(spacing = 0.5*angstrom)
program.set_eos('MBWR')
program.init_free_energy(temperature)

program.fener.add_external_potential(temperature)
program.fener.add_hard_sphere()
program.fener.add_mean_field()
program.fener.add_correlation_wda_lj()
program.set_temperature(temperature)

# A converged density profile is required before extracting adsorption data.
# The Anderson mixer is used here to accelerate the nonlinear self-consistent iteration.
criteria = ['riue', 'res']
thresholds = [1e-5, 1e-6]

solver = Anderson(program, criterion=criteria, threshold=thresholds)
program.set_solver()


In [ ]:
# Solve the adsorption problem over the requested pressure range.
# This performs the equilibrium calculation and writes the results into the OutputFiles folder.
program.adsorption_isotherm(temperature, pressures, rewrite=True)


In [ ]:
# Extract the adsorption loading from the completed calculation.
# The calculator wraps the program state and converts the solved density profile into loadings.
calc = Calculator(program)
loadings = calc.return_loading(300, pressures=pressures)
print('Loadings at the selected pressures [mol/kg]:', loadings)


# Guest mixture

The next part of the notebook repeats the same workflow for a binary guest mixture. This is useful when the adsorbate is a mixture of species with different molecular parameters and composition, and the equilibrium loading must account for both species simultaneously.

The `GuestMixture` object bundles multiple guest definitions and their mole fractions, which are then used to construct the multi-component adsorption problem.


This example is performed with PC-SAFT functional, however, it is also entirely possible to perform mixture calculations with MFA-WDAc

In [ ]:
# Define the first guest species: ethane
guestname1 = 'C2H6_UA'
mass1, sigma1, epsilon1, m1 = SAFT_pars_dict['C2H6']
guestchk1 = 'InputFiles/Guest/C2H6/struct_CA.chk'
guestpar1 = 'InputFiles/Guest/C2H6/pars_vdw_CA_trappe.txt'
guest1 = DualModelGuest(guestname1, mass1, sigma1, epsilon1, struct=guestchk1, par=guestpar1, m=m1, ffname='TRAPPE-PCSAFT', hs_def='exp')

# Define the second guest species: ethylene
guestname2 = 'C2H4_UA'
mass2, sigma2, epsilon2, m2 = SAFT_pars_dict['C2H4']
guestchk2 = 'InputFiles/Guest/C2H4/struct_CA.chk'
guestpar2 = 'InputFiles/Guest/C2H4/pars_vdw_CA_trappe.txt'
guest2 = DualModelGuest(guestname2, mass2, sigma2, epsilon2, struct=guestchk2, par=guestpar2, m=m2, ffname='TRAPPE-PCSAFT', hs_def='exp')

# The mixture composition is provided as mole fractions for the guest species.
# These fractions are used when building the guest-mixture thermodynamic model.
mole_fractions = [0.5, 0.5]
mixture = GuestMixture([guest1, guest2], mole_fractions)

hostname = 'ZIF8'
hostchk = 'InputFiles/Host/ZIF8/ZIF8_struct.chk'
hostpar = 'InputFiles/Host/ZIF8/pars_vdw_AMBER.txt'
host = NanoporousHost(hostname, hostchk, hostpar)

ff_suffix = 'AMBER_TRAPPE'
funct_suffix = 'tWBII_PCSAFT'

grid_suffix = '0_50'


In [ ]:
# program initialization for the guest mixture
# The same program architecture is used, but the guest is now a mixture object rather than a single guest.
program = Program(prefix=f'OutputFiles/AdsorptionExample', hostname=hostname, guestname=mixture.name, ff_suffix=ff_suffix, funct_suffix=funct_suffix, grid_suffix=grid_suffix)

program.set_system(host, guest)
program.set_grid(spacing = 0.5*angstrom)
program.set_eos('PCSAFT')
program.init_free_energy(temperature)

program.fener.add_external_potential(temperature)
program.fener.add_hard_sphere()
program.fener.add_PCSAFT()
program.set_temperature(temperature)

criteria = ['riue', 'res']
thresholds = [1e-5, 1e-6]

solver = Anderson(program, criterion=criteria, threshold=thresholds)
program.set_solver()


In [ ]:
# Solve the mixture adsorption isotherm using the same strategy as for a single guest.
# In practice, this computes the equilibrium density profile for each pressure in the mixture.
program.adsorption_isotherm(temperature, pressures, rewrite=True)


In [ ]:
calc = Calculator(program)
plotter = Plotter(calc)